In [1]:
import pyspark 
print(pyspark.__version__)

3.5.0


In [2]:
from pyspark.sql import SparkSession 

spark2 = (
    SparkSession
    .builder 
    .appName("Reading parquet data from Minio")
    .master("spark://spark-master:7077")
    .config("spark.cores.max", "4")
    .config(
        "org.postgresql:postgresql:42.7.3",
        "spark.sql.shuffle.partitions, 8"
    )
    .getOrCreate()
)
spark2

In [3]:
spark2.conf.set("fs.s3a.endpoint", "http://minio:9000")
spark2.conf.set("fs.s3a.access.key", "minioadmin")
spark2.conf.set("fs.s3a.secret.key", "minioadmin")
spark2.conf.set("fs.s3a.path.style.access", "true")
spark2.conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
spark2.conf.set("fs.s3a.aws.credentials.provider",
               "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")

In [4]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import *

schema_listen_events = StructType([
    StructField("artist",        StringType(),  True),
    StructField("song",          StringType(),  True),
    StructField("duration",      DoubleType(),  True),
    StructField("ts",            LongType(),    True),
    StructField("sessionId",     IntegerType(), True),
    StructField("auth",          StringType(),  True),
    StructField("level",         StringType(),  True),
    StructField("itemInSession", IntegerType(), True),
    StructField("city",          StringType(),  True),
    StructField("zip",           StringType(),  True),
    StructField("state",         StringType(),  True),
    StructField("userAgent",     StringType(),  True),
    StructField("lon",           DoubleType(),  True),
    StructField("lat",           DoubleType(),  True),
    StructField("userId",        IntegerType(), True),
    StructField("lastName",      StringType(),  True),
    StructField("firstName",     StringType(),  True),
    StructField("gender",        StringType(),  True),
    StructField("registration",  LongType(),    True),
])

schema_status_change_events = StructType([
    StructField("ts",            LongType(),    True),
    StructField("sessionId",     IntegerType(), True),
    StructField("level",         StringType(),  True),
    StructField("itemInSession", IntegerType(), True),
    StructField("city",          StringType(),  True),
    StructField("zip",           StringType(),  True),
    StructField("state",         StringType(),  True),
    StructField("userAgent",     StringType(),  True),
    StructField("lon",           DoubleType(),  True),
    StructField("lat",           DoubleType(),  True),
    StructField("userId",        IntegerType(), True),
    StructField("lastName",      StringType(),  True),
    StructField("firstName",     StringType(),  True),
    StructField("gender",        StringType(),  True),
    StructField("registration",  LongType(),    True),
    StructField("success",       BooleanType(), True),
])

schema_auth_events = StructType([
    StructField("ts",            LongType(),    True),
    StructField("sessionId",     IntegerType(), True),
    StructField("level",         StringType(),  True),
    StructField("itemInSession", IntegerType(), True),
    StructField("city",          StringType(),  True),
    StructField("zip",           StringType(),  True),
    StructField("state",         StringType(),  True),
    StructField("userAgent",     StringType(),  True),
    StructField("lon",           DoubleType(),  True),
    StructField("lat",           DoubleType(),  True),
    StructField("success",       BooleanType(), True),
])

schema_page_view_events = StructType([
    StructField("ts",            LongType(),    True),
    StructField("sessionId",     IntegerType(), True),
    StructField("page",          StringType(),  True),
    StructField("auth",          StringType(),  True),
    StructField("method",        StringType(),  True),
    StructField("status",        IntegerType(), True),
    StructField("level",         StringType(),  True),
    StructField("itemInSession", IntegerType(), True),
    StructField("city",          StringType(),  True),
    StructField("zip",           StringType(),  True),
    StructField("state",         StringType(),  True),
    StructField("userAgent",     StringType(),  True),
    StructField("lon",           DoubleType(),  True),
    StructField("lat",           DoubleType(),  True),
    StructField("userId",        IntegerType(), True),
    StructField("lastName",      StringType(),  True),
    StructField("firstName",     StringType(),  True),
    StructField("gender",        StringType(),  True),
    StructField("registration",  LongType(),    True),
    StructField("artist",        StringType(),  True),
    StructField("song",          StringType(),  True),
    StructField("duration",      DoubleType(),  True),
])


In [5]:
# def postgres(df, table_name):
#     (
# 	df.write
#     	.mode("append")
#     	.format("jdbc")
#     	.option("driver", "org.postgresql.Driver")
#     	.option("url", "jdbc:postgresql://postgres-db:5432/sqlpad")
#     	.option("dbtable", f"raw.{table_name}")
#     	.option("user", "sqlpad")
#     	.option("password", "sqlpad")
#     	.save()
#     )
def postgres(df, table_name):

    (
        df.write
        .mode("append")
        .format("jdbc")
        .option("driver", "org.postgresql.Driver")
        .option("url", "jdbc:postgresql://postgres-db:5432/sqlpad")
        .option("dbtable", f"raw.{table_name}")
        .option("user", "sqlpad")
        .option("password", "sqlpad")
        .option("batchsize", 5000)
        .option("reWriteBatchedInserts", "true")
        .save()
    )

In [6]:
# df_stream = spark2.readStream \
#     .format("parquet") \
#     .schema(schema_listen_events) \
#     .load("s3a://listen-events/data/")

# def write_to_postgres(batch_df, batch_id):
#     print(f"Batch_id: {batch_id}, rows: {batch_df.count()}")
#     if batch_df.count() > 0:
#         postgres(batch_df, "listen_events")
#         batch_df.show(10, truncate=False)

# query = df_stream.writeStream \
#     .foreachBatch(write_to_postgres) \
#     .trigger(processingTime="15 seconds") \
#     .option("checkpointLocation", "s3a://listen-events/checkpoints-postgres/") \
#     .start()

# query.awaitTermination()

def start_stream(bucket, schema, table):
    print(f"Table: {table}")

    df_stream = spark2.readStream \
        .format("parquet") \
        .schema(schema) \
        .load(f"s3a://{bucket}/data/")

    def write_batch(batch_df, batch_id):

        rows = batch_df.count()
        print(f"[{table}] batch {batch_id}, rows: {rows}")
        batch_df.show(10, truncate=False)

        if rows > 0:
            postgres(batch_df, table)

    query = df_stream.writeStream \
        .foreachBatch(write_batch) \
        .trigger(processingTime="15 seconds") \
        .option("checkpointLocation", f"s3a://{bucket}/checkpoint_{table}/") \
        .start()

    return query

q1 = start_stream("auth-events", schema_auth_events, "auth_events")
q2 = start_stream("listen-events", schema_listen_events, "listen_events")
q3 = start_stream("page-view-events", schema_page_view_events, "page_view_events")
q4 = start_stream("status-change-events", schema_status_change_events, "status_change_events")

spark2.streams.awaitAnyTermination()

Table: auth_events
Table: listen_events
Table: page_view_events
Table: status_change_events
[auth_events] batch 0, rows: 20481
[listen_events] batch 0, rows: 1402332
[status_change_events] batch 0, rows: 969
[page_view_events] batch 0, rows: 1640642
+-------------+---------+-----+-------------+-----------------+-----+-----+--------------------------------------------------------------------------------------------------------------------------+----------+---------+------+--------+---------+------+-------------+-------+
|ts           |sessionId|level|itemInSession|city             |zip  |state|userAgent                                                                                                                 |lon       |lat      |userId|lastName|firstName|gender|registration |success|
+-------------+---------+-----+-------------+-----------------+-----+-----+--------------------------------------------------------------------------------------------------------------------------+--

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 